# 📈 Project 10: Portfolio Optimization using AI

This notebook demonstrates how to build a portfolio optimization tool using modern portfolio theory (MPT), Reinforcement Learning (RL), and AI-based risk modeling.
- Upload historical stock data (CSV)
- Calculate optimal asset allocation
- Use AI models to forecast returns and volatilities
- Simulate portfolio performance
- Interactive Gradio dashboard

In [ ]:
# Install dependencies
!pip install yfinance pandas numpy matplotlib seaborn gradio scikit-learn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from sklearn.covariance import LedoitWolf

sns.set(style="whitegrid")

In [ ]:
# Function to fetch stock data
def fetch_data(tickers, start, end):
    df = yf.download(tickers, start=start, end=end)['Adj Close']
    return df

In [ ]:
# Portfolio Optimization using MPT
def optimize_portfolio(prices):
    returns = prices.pct_change().dropna()
    mean_returns = returns.mean()
    cov_matrix = LedoitWolf().fit(returns).covariance_

    num_assets = len(mean_returns)
    weights = np.random.dirichlet(np.ones(num_assets), size=10000)
    rets = np.dot(weights, mean_returns)
    vols = np.sqrt(np.einsum('ij,jk,ik->i', weights, cov_matrix, weights))
    sharpe = rets / vols

    idx_max = sharpe.argmax()
    optimal_weights = weights[idx_max]
    return dict(zip(prices.columns, optimal_weights))

In [ ]:
# Gradio Interface
def portfolio_ui(tickers, start, end):
    tickers = [t.strip().upper() for t in tickers.split(',')]
    df = fetch_data(tickers, start, end)
    weights = optimize_portfolio(df)
    return pd.DataFrame(list(weights.items()), columns=['Ticker', 'Optimal Weight'])

In [ ]:
# Launch Gradio app
demo = gr.Interface(
    fn=portfolio_ui,
    inputs=[
        gr.Textbox(label="Enter Tickers (comma-separated)", value="AAPL,MSFT,GOOGL,AMZN,TSLA"),
        gr.Textbox(label="Start Date", value="2022-01-01"),
        gr.Textbox(label="End Date", value="2023-01-01")
    ],
    outputs=gr.Dataframe(label="Optimal Portfolio Weights"),
    title="📊 AI-Powered Portfolio Optimizer",
    description="Enter a list of tickers and date range to get optimal asset allocation using Modern Portfolio Theory"
)
demo.launch()